In [4]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *

In [5]:
model = "Llama-3-8B-Instruct_4_shot"

### Model Output to nice JSON and Failure 

In [6]:
input_dir = f"model_output/{model}/output/"
output_dir = f"model_output/{model}/formatted/"
failure_dir = f"model_output/{model}/failed/"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
if not os.path.exists(failure_dir):
    os.makedirs(failure_dir)

for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, filename)
        failure_file_path = os.path.join(failure_dir, filename)
        try:
            file = read_json(input_file_path)
            print(f"Processing file: {filename}")
            save_json_to_file(file, output_file_path)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            shutil.move(input_file_path, failure_file_path)

Processing file: Llama-3-8B-Instruct_NCT00050349_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00050349_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00094861_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00094861_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00122070_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00122070_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00198913_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00198913_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00235170_exc_4_shot.json
Processing file: Llama-3-

In [7]:
p2_label_path = "chia_label/p2"
ready_path = f"model_output/{model}/ready"
failed_model_path = f"model_output/{model}/failed_inner"
p2_model_formatted_path = f"model_output/{model}/formatted"
eval_path = f"evaluate/batch1"

for path in [ready_path, failed_model_path, p2_model_formatted_path]: 
    os.makedirs(path, exist_ok=True)

In [13]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

def extract_raw_texts(data):
    raw_texts = set()
    def traverse(node):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == "raw_text":
                    raw_texts.add(value)
                else:
                    traverse(value)
        elif isinstance(node, list):
            for item in node:
                traverse(item)

    traverse(data)
    return raw_texts

def extract_words(raw_texts):
    words = set()
    for text in raw_texts:
        words.update(text.split())
    return words

In [14]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [15]:
labels = []
predictions = []
success_data = []

In [20]:
for nct in common_ncts:# ["NCT00050349_exc"]
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])
        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)

        # Try count words
        label_raw_texts = extract_raw_texts(label_data)
        model_raw_texts = extract_raw_texts(model_data)
        missing_texts = label_raw_texts - model_raw_texts
        label_words = extract_words(label_raw_texts)
        model_words = extract_words(model_raw_texts)
        missing_words = label_words - model_words
        missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
        print(missing_words_pct)
        
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0),
            'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
            'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
            'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
            'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0
            
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

Error processing NCT NCT00965900_exc: Expecting ',' delimiter: line 74 column 2 (char 1921)
24.444444444444443
59.130434782608695
17.5
8.0
13.333333333333334
18.75
0.0
0.0
0.0
0.0
12.871287128712872
0.0
60.71428571428571
33.33333333333333
5.263157894736842
0.0
7.017543859649122
0.0
0.0
6.451612903225806
18.181818181818183
0.0
31.46067415730337
39.784946236559136
30.434782608695656
4.761904761904762
Error processing NCT NCT01217671_inc: Extra data: line 110 column 6 (char 3808)
75.98684210526315
0.0
13.043478260869565
8.620689655172415
Error processing NCT NCT01032109_exc: Extra data: line 42 column 1 (char 772)
0.0
9.523809523809524
5.714285714285714
0.0
69.37269372693727
3.6363636363636362
8.73015873015873
Error processing NCT NCT00785213_exc: Extra data: line 108 column 1 (char 3110)
5.555555555555555
14.285714285714285
3.7037037037037033
0.0
0.0
1.4492753623188406
34.5679012345679
10.526315789473683
0.0
Error processing NCT NCT01214096_exc: Expecting ',' delimiter: line 161 column 2

In [44]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

In [45]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

In [46]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows